<a href="https://colab.research.google.com/github/maryamyaser/ATM-System/blob/main/AMAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
!{sys.executable} -m pip install -q catboost

# Importing Libraries

In [ ]:
# Main Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from matplotlib import ticker
import scipy.stats as stats_lib
from scipy.stats import probplot
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor,VotingRegressor,StackingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import (
   mean_absolute_error, mean_squared_error, r2_score
)
import warnings
import os


warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option("display.float_format", lambda x: "{:,.2f}".format(x))
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

In [ ]:
df = pd.read_parquet("/content/Food_Prices_in_Egypt.parquet")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info

## Data Cleaning 🧹

In this section, we identify and handle missing values, duplicate records, incorrect data types, and invalid values to ensure that the dataset is clean and ready for analysis and modeling.

In [ ]:
df.isnull().sum()

In [ ]:
missing_df = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Pct (%)': (df.isnull().mean()) * 100
})

missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values(by='Missing_Count', ascending=False)
missing_df

In [ ]:
missing_columns = missing_df.index.tolist()

df[missing_columns].describe()

In [ ]:
df[
    [
        "Supply_Level",
        "Demand_Level",
        "Transport_Cost"
    ]
].corr()

### Missing Values

In [ ]:
# Fill Supply with median
df["Supply_Level"] = df["Supply_Level"].fillna(
    df["Supply_Level"].median()
)

In [ ]:
#Fill Demand with median
df["Demand_Level"] = df["Demand_Level"].fillna(
    df["Demand_Level"].median()
)

In [ ]:
#Fill Demand with median
df["Transport_Cost"] = df["Transport_Cost"].fillna(
    df["Transport_Cost"].median()
)

In [ ]:
df[
    [
        "Supply_Level",
        "Demand_Level",
        "Transport_Cost"
    ]
].isnull().sum()

In [ ]:
df["Supply_Demand_Gap"] = df["Supply_Level"] - df["Demand_Level"]
df["Supply_Demand_Ratio"] = df["Supply_Level"] / df["Demand_Level"]

In [ ]:
df[["Transport_Cost"]].describe()

### Duplicates

In [ ]:
# Check for duplicate rows
duplicate_count = df.duplicated().sum()
print("Number of duplicate rows:", duplicate_count)

### Data Types

In [ ]:
# Convert Date to datetime
df["Date"] = pd.to_datetime(df["Date"])

In [ ]:
print(df["Date"].dtype)

In [ ]:
# Check invalid prices
invalid_price = (df["Price_EGP"] <= 0).sum()
print("Invalid Price_EGP values:", invalid_price)

In [ ]:
# Check Supply range
invalid_supply = (
    (df["Supply_Level"] < 0) | (df["Supply_Level"] > 100)).sum()

In [ ]:
print("Invalid Supply_Level values:", invalid_supply)

In [ ]:
# Check demand range
invalid_demand = (
    (df["Demand_Level"] < 0) | (df["Demand_Level"] > 100)).sum()

In [ ]:
print("Invalid Demand_Level values:", invalid_demand)

In [ ]:
# Check Is_Ramadan values
print(df["Is_Ramadan"].value_counts())

In [ ]:
# Final cleaning check
print("Missing values:", df.isnull().sum().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nData shape:", df.shape)

# Exploratory Data Analysis (EDA) 📊

The goal of this section is to explore food price patterns and identify relationships between prices, commodities, locations, time, supply, and demand.

### Understanding the Categorical Variables

In [ ]:
categorical_columns = df.select_dtypes(include="category").columns

categorical_columns

In [ ]:
for col in categorical_columns:
    print(f"\n{col}:")
    print("Number of unique values:", df[col].nunique())
    print(df[col].unique())

In [ ]:
numeric_columns = df.select_dtypes(include="number").columns

In [ ]:
datetime_columns = df.select_dtypes(include="datetime").columns

### Commodity Distribution

We first explore the distribution of food commodities in the dataset to understand which products are represented and how frequently they occur.

In [ ]:
commodity_counts = df["Commodity"].value_counts()
commodity_counts

In [ ]:
plt.figure(figsize=(18, 6))

bars = plt.bar(
    commodity_counts.index,
    commodity_counts.values,
    color=plt.cm.viridis(
        np.linspace(0.15, 0.9, len(commodity_counts))
    )
)

plt.title(
    "Number of Records per Commodity",
    fontsize=16,
    fontweight="bold",
    pad=15
)

plt.xlabel("Commodity", fontsize=11)
plt.ylabel("Number of Records", fontsize=11)

# Make names smaller and slightly spaced
plt.xticks(
    rotation=0,
    fontsize=7
)

plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)

plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.25
)

# Values
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height + 15,
        f"{int(height)}",
        ha="center",
        va="bottom",
        fontsize=7
    )

plt.tight_layout()
plt.show()

In [ ]:
# Top 5 commodities by number of records
print("Top 5 commodities:")
print(commodity_counts.head(5))

print("\nBottom 5 commodities:")
print(commodity_counts.tail(5))

### Governorate Distribution

In [ ]:
governorate_counts = df["Governorate"].value_counts()
governorate_counts

In [ ]:
governorate_counts = df["Governorate"].value_counts()
plt.figure(figsize=(14, 6))

bars = plt.bar(
    governorate_counts.index,
    governorate_counts.values,
    color=plt.cm.viridis(
        np.linspace(0.15, 0.9, len(governorate_counts))
    )
)

plt.title(
    "Number of Records per Governorate",
    fontsize=16,
    fontweight="bold",
    pad=15
)

plt.xlabel("Governorate", fontsize=11)
plt.ylabel("Number of Records", fontsize=11)

plt.xticks(
    rotation=0,
    fontsize=8
)

# Remove unnecessary borders
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)

# Light grid
plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.25
)

# Add values on top
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height + 10,
        f"{int(height)}",
        ha="center",
        va="bottom",
        fontsize=8
    )

plt.tight_layout()
plt.show()

In [ ]:
print("Top 5 Governorates:")
print(governorate_counts.head(5))

print("\nBottom 5 Governorates:")
print(governorate_counts.tail(5))

### Price Analysis

#### Overall Price Distribution

In [ ]:
price_stats = df["Price_EGP"].describe()
price_stats

In [ ]:
plt.figure(figsize=(12, 6))

plt.hist(
    df["Price_EGP"].dropna(),
    bins=50,
    color=plt.cm.viridis(0.6),
    edgecolor="white",
    linewidth=0.7
)

plt.title(
    "Distribution of Food Prices",
    fontsize=16,
    fontweight="bold",
    pad=15
)

plt.xlabel("Price (EGP)", fontsize=11)
plt.ylabel("Frequency", fontsize=11)

# Remove unnecessary borders
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)

# Light grid
plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.25
)

plt.tight_layout()
plt.show()

### Insight

- The food price distribution is strongly right-skewed, with most observations concentrated at relatively lower prices.
- A long right tail can be observed, indicating the presence of some very high price observations.
- These high values may represent potential outliers, but they should not be removed automatically because they may reflect genuine market price differences.

#### Price Outlier Analysis

In [ ]:
Q1 = df["Price_EGP"].quantile(0.25)
Q3 = df["Price_EGP"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

In [ ]:
outliers = df[(df["Price_EGP"] < lower_bound) |(df["Price_EGP"] > upper_bound)]
print("Number of outliers:", len(outliers))
print("Outlier percentage:", len(outliers) / len(df) * 100)

In [ ]:
plt.figure(figsize=(12, 4))

plt.boxplot(
    df["Price_EGP"].dropna(),
    vert=False,
    patch_artist=True,
    boxprops=dict(
        facecolor=plt.cm.viridis(0.6),
        edgecolor="black"
    ),
    medianprops=dict(
        color="black",
        linewidth=2
    ),
    whiskerprops=dict(
        color="black"
    ),
    capprops=dict(
        color="black"
    ),
    flierprops=dict(
        marker="o",
        markerfacecolor=plt.cm.viridis(0.8),
        markeredgecolor="black",
        markersize=4,
        alpha=0.5
    )
)

plt.title(
    "Distribution of Food Prices",
    fontsize=16,
    fontweight="bold",
    pad=15
)

plt.xlabel("Price (EGP)", fontsize=11)

# Clean appearance
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.gca().spines["left"].set_visible(False)

plt.grid(
    axis="x",
    linestyle="--",
    alpha=0.25
)

plt.tight_layout()
plt.show()

### Insight

- The boxplot confirms the presence of a number of high-price observations beyond the upper whisker.
- Using the IQR method, 25,956 observations (5.31% of the dataset) were identified as potential outliers.
- These observations are retained because they may represent genuine differences in food prices rather than data errors.

#### Price Distribution by Commodity

In [ ]:
commodity_price_stats = (df.groupby("Commodity", observed=True)["Price_EGP"].agg(["count", "mean", "median", "min", "max"]).sort_values("median", ascending=False))
commodity_price_stats

In [ ]:
plt.figure(figsize=(12, 10))

commodity_median = (
    df.groupby("Commodity")["Price_EGP"]
    .median()
    .sort_values()
    .index
)

sns.boxplot(
    data=df,
    y="Commodity",
    x="Price_EGP",
    order=commodity_median,
    palette="viridis",
    width=0.6,
    fliersize=3
)

plt.title(
    "Price Distribution by Commodity",
    fontsize=18,
    fontweight="bold",
    pad=15
)

plt.xlabel("Price (EGP)", fontsize=12)
plt.ylabel("Commodity", fontsize=12)

plt.grid(
    axis="x",
    linestyle="--",
    alpha=0.25
)

plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
median_prices = (
    df.groupby("Commodity", observed=True)["Price_EGP"]
      .median()
      .sort_values(ascending=True)
)

plt.figure(figsize=(12, 9))

bars = plt.barh(
    median_prices.index,
    median_prices.values,
    color=plt.cm.viridis(
        (median_prices.values - median_prices.values.min()) /
        (median_prices.values.max() - median_prices.values.min())
    )
)

plt.title(
    "Median Food Price by Commodity",
    fontsize=17,
    fontweight="bold",
    pad=15
)

plt.xlabel("Median Price (EGP)", fontsize=11)
plt.ylabel("Commodity", fontsize=11)

# Add values
for bar, value in zip(bars, median_prices.values):
    plt.text(
        value + median_prices.max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.0f}",
        va="center",
        fontsize=9
    )

plt.grid(
    axis="x",
    linestyle="--",
    alpha=0.25
)

plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

### Price Analysis by Governorate

#### Average and Median Price by Governorate

In [ ]:
governorate_price_stats = (
    df.groupby("Governorate", observed=True)["Price_EGP"]
      .agg(["count", "mean", "median", "min", "max"])
      .sort_values("median", ascending=False)
)

governorate_price_stats

In [ ]:
print("Top 5 Governorates by Median Price:")
print(governorate_price_stats["median"].head(5))

print("\nBottom 5 Governorates by Median Price:")
print(governorate_price_stats["median"].tail(5))

In [ ]:
governorate_price_stats = (
    df.groupby("Governorate", observed=True)["Price_EGP"]
      .agg(["count", "mean", "median", "min", "max"])
      .sort_values("median", ascending=False)
)

median_governorate = governorate_price_stats["median"].sort_values()

plt.figure(figsize=(12, 8))

bars = plt.barh(
    median_governorate.index,
    median_governorate.values,
    color=plt.cm.viridis(
        (median_governorate.values - median_governorate.values.min()) /
        (median_governorate.values.max() - median_governorate.values.min())
    )
)

plt.title(
    "Median Food Price by Governorate",
    fontsize=17,
    fontweight="bold",
    pad=15
)

plt.xlabel("Median Price (EGP)", fontsize=11)
plt.ylabel("Governorate", fontsize=11)

# Add values
for bar, value in zip(bars, median_governorate.values):
    plt.text(
        value + median_governorate.max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.0f}",
        va="center",
        fontsize=9
    )

plt.grid(
    axis="x",
    linestyle="--",
    alpha=0.25
)

plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
price_heatmap = (
    df.pivot_table(
        index="Governorate",
        columns="Commodity",
        values="Price_EGP",
        aggfunc="median",
        observed=True
    )
)

plt.figure(figsize=(16, 9))

sns.heatmap(
    price_heatmap,
    cmap="viridis",
    annot=False
)

plt.title(
    "Median Food Price by Governorate and Commodity",
    fontsize=17,
    fontweight="bold",
    pad=15
)

plt.xlabel("Commodity")
plt.ylabel("Governorate")

plt.tight_layout()
plt.show()

In [ ]:
monthly_price = (
    df.groupby("Date", observed=True)["Price_EGP"]
      .median()
)

plt.figure(figsize=(14, 6))

plt.plot(
    monthly_price.index,
    monthly_price.values,
    linewidth=2
)

plt.title(
    "Food Price Trend Over Time",
    fontsize=17,
    fontweight="bold",
    pad=15
)

plt.xlabel("Date")
plt.ylabel("Median Price (EGP)")

plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.25
)

plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
monthly_seasonality = (
    df.groupby("Month", observed=True)["Price_EGP"]
      .median()
)

plt.figure(figsize=(10, 6))

plt.plot(
    monthly_seasonality.index,
    monthly_seasonality.values,
    marker="o",
    linewidth=2
)

plt.title(
    "Median Food Price by Month",
    fontsize=17,
    fontweight="bold",
    pad=15
)

plt.xlabel("Month")
plt.ylabel("Median Price (EGP)")

plt.xticks(range(1, 13))

plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.25
)

plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
ramadan_prices = (
    df.groupby("Is_Ramadan", observed=True)["Price_EGP"]
      .median()
)

plt.figure(figsize=(8, 5))

bars = plt.bar(
    ["Non-Ramadan", "Ramadan"],
    ramadan_prices.values
)

plt.title(
    "Median Food Price: Ramadan vs Non-Ramadan",
    fontsize=17,
    fontweight="bold",
    pad=15
)

plt.ylabel("Median Price (EGP)")

for bar, value in zip(bars, ramadan_prices.values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value,
        f"{value:.0f}",
        ha="center",
        va="bottom"
    )

plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
numeric_cols = [
    "Price_EGP",
    "Transport_Cost",
    "Inflation_Index",
]

corr = df[numeric_cols].corr()

plt.figure(figsize=(10, 7))

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="viridis",
    center=0
)

plt.title(
    "Correlation Matrix of Numerical Features",
    fontsize=17,
    fontweight="bold",
    pad=15
)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(20, 20))
sns.heatmap(df.corr(numeric_only=True), annot=True,fmt='.2f', cmap='coolwarm')
plt.title("Feature Correlation Matrix")
plt.show()

In [ ]:
df.isnull().sum()

# Dropping Unnecessary Columns 🗑️

In this section, we drop columns that are either data leakage sources (features derived from future or target-related information), identifiers not useful for modeling, or redundant with other existing features.

In [ ]:
#drop some features that start with same target
columns_to_drop = [
    'Series_ID', 'Date',
    'Expected_Price', 'Expected_Lower', 'Expected_Upper',
    'Observed_User_Price', 'Price_Deviation_Percent', 'Price_ZScore',
    'Is_Anomaly', 'Price_Status','Price_Changed per 3days',
    'Target_Price_1D','Target_Price_Change_1D_Pct', 'Target_Price_Direction',
    'Price_Lag_1', 'Price_Lag_2', 'Price_Lag_7', 'Price_Lag_8',
    'Price_Change_1D_Pct', 'Price_Change_7D_Pct',
    'Rolling_Mean_7', 'Rolling_Std_7', 'Rolling_Mean_30', 'Rolling_Std_30',
    'Historical_Min', 'Historical_Max',
    'Quarter', 'Week', 'DayOfWeek', 'DayOfYear',
    'Latitude', 'Longitude', 'Distance_From_Cairo_KM', 'Logistics_Score',
    'Supply_Demand_Ratio', 'Market','Supply_Demand_Gap'
]

df= df.drop(columns=[col for col in columns_to_drop if col in df.columns])

print("=:", df.shape)
print("\n:")
print(df.columns.tolist())




In [ ]:
#display data
df.head()


# Encoding Categorical Variables 🔤

In this section, we convert categorical (text-based) columns into numerical format using One-Hot Encoding, so that the machine learning model can process them.




In [ ]:
df_encoded = pd.get_dummies(df, columns=['Season','Governorate','Region','Commodity','Market_Type','Category','Unit'], drop_first=True)
df_encoded.head()


# Data Cleaning: Duplicate Check 🔍

In this section, we check whether the dataset contains any duplicate rows, to ensure the data quality before proceeding with feature engineering and modeling.

In [ ]:
print("duplicated columns", df.duplicated().sum())

# Train / Validation / Test Split 🧩

In this section, we split the dataset into training, validation, and testing sets based on the predefined Split column, to properly evaluate model performance.

In [ ]:
#split data
train_df = df_encoded [df["Split"] == "Train"].copy()
valid_df = df_encoded [df["Split"] == "Validation"].copy()
test_df = df_encoded [df["Split"] == "Test"].copy()

print(train_df.shape , valid_df.shape , test_df.shape)

In [ ]:
target_column = 'Price_EGP'

X_train = train_df.drop(columns=[target_column, 'Split'], errors='ignore')
y_train = train_df[target_column]

X_val = valid_df.drop(columns=[target_column, 'Split'], errors='ignore')
y_val = valid_df[target_column]

X_test = test_df.drop(columns=[target_column, 'Split'], errors='ignore')
y_test = test_df[target_column]

# Feature Scaling 📏

In this section, we standardize the numerical features using StandardScaler, so that all features are on the same scale and the model can learn effectively.

In [ ]:
scalar=StandardScaler()
X_train_scaled= scalar.fit_transform(X_train)
X_val_scaled= scalar.transform(X_val)
X_test_scaled= scalar.transform(X_test)
pd.DataFrame(X_train_scaled, columns=X_train.columns).head()



#Training models🌡🌠

In [ ]:
#linear regression
lr_model=LinearRegression()
lr_model.fit(X_train_scaled,y_train)

lr_pred=lr_model.predict(X_val_scaled)
lr_mae=mean_absolute_error(y_val,lr_pred)
lr_mse=mean_squared_error(y_val,lr_pred)
lr_r2=r2_score(y_val,lr_pred)

print(f"\n--- Linear Regression (Validation) ---")
print(f"MAE: {lr_mae:.3f}")
print(f"MSE: {lr_mse:.3f}")
print(f"R2 : {lr_r2:.3f}")




In [ ]:

y_train_pred = lr_model.predict(X_train_scaled)
y_val_pred = lr_model.predict(X_val_scaled)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_train, y_train_pred, color='blue', alpha=0.3)
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
plt.xlabel("Actual (Train)")
plt.ylabel("Predicted (Train)")
plt.title("linear regression: Training Data")

plt.subplot(1, 2, 2)
plt.scatter(y_val, y_val_pred, color='green', alpha=0.3)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', lw=2)
plt.xlabel("Actual (Validation)")
plt.ylabel("Predicted (Validation)")
plt.title("linear regression: Validation Data")
plt.show()

plt.tight_layout()
plt.show()

In [ ]:
sample_size = 10000

rf_model = RandomForestRegressor(n_estimators=20, random_state=42, n_jobs=-1)

rf_model.fit(X_train_scaled[:sample_size], y_train[:sample_size])

rf_pred = rf_model.predict(X_val_scaled)
rf_mae = mean_absolute_error(y_val, rf_pred)
rf_mse = mean_squared_error(y_val, rf_pred)
rf_r2 = r2_score(y_val, rf_pred)

print(f"\n--- Random Forest (Validation) ---")
print(f"MAE: {rf_mae:.3f}")
print(f"MSE: {rf_mse:.3f}")
print(f"R2 : {rf_r2:.3f}")

In [ ]:

sample_size = 1000

y_train_pred = rf_model.predict(X_train_scaled[:sample_size])
y_val_pred = rf_model.predict(X_val_scaled[:sample_size])

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_train[:sample_size], y_train_pred, color='blue', alpha=0.3)
plt.plot([y_train[:sample_size].min(), y_train[:sample_size].max()],
         [y_train[:sample_size].min(), y_train[:sample_size].max()], 'r--', lw=2)
plt.xlabel("Actual (Train Sample)")
plt.ylabel("Predicted (Train Sample)")
plt.title("Random Forest: Training Sample Data")

plt.subplot(1, 2, 2)
plt.scatter(y_val[:sample_size], y_val_pred, color='green', alpha=0.3)
plt.plot([y_val[:sample_size].min(), y_val[:sample_size].max()],
         [y_val[:sample_size].min(), y_val[:sample_size].max()], 'r--', lw=2)
plt.xlabel("Actual (Val Sample)")
plt.ylabel("Predicted (Val Sample)")
plt.title("Random Forest: Validation Sample Data")

plt.tight_layout()
plt.show()

In [ ]:
def evaluate_model(model, X_train, y_train, X_val, y_val, name):
    model.fit(X_train, y_train)
    pred = model.predict(X_val)

    mae = mean_absolute_error(y_val, pred)
    mse = mean_squared_error(y_val, pred)
    r2 = r2_score(y_val, pred)

    print(f"\n--- {name} (Validation) ---")
    print(f"MAE: {mae:.3f}")
    print(f"MSE: {mse:.3f}")
    print(f"R2 : {r2:.3f}")

    return {"Model": name, "MAE": mae, "MSE": mse, "R2": r2}


In [ ]:
results = []

In [ ]:
knn_model = KNeighborsRegressor(n_neighbors=5)
results.append(evaluate_model(knn_model, X_train_scaled, y_train, X_val_scaled, y_val, "KNN"))

In [ ]:
y_train_pred = knn_model.predict(X_train_scaled)
y_val_pred = knn_model.predict(X_val_scaled)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_train, y_train_pred, color='blue', alpha=0.3)
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
plt.xlabel("Actual (Train)")
plt.ylabel("Predicted (Train)")
plt.title("knn_model: Training Data")

plt.subplot(1, 2, 2)
plt.scatter(y_val, y_val_pred, color='green', alpha=0.3)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', lw=2)
plt.xlabel("Actual (Validation)")
plt.ylabel("Predicted (Validation)")
plt.title("knn_model: Validation Data")
plt.show()

In [ ]:
# 2) Decision Tree Regressor
dt_model = DecisionTreeRegressor(max_depth=10, random_state=42)
results.append(evaluate_model(dt_model, X_train_scaled, y_train, X_val_scaled, y_val, "Decision Tree"))

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

plt.figure(figsize=(20, 10))
plot_tree(dt_model,

          filled=True,
          max_depth=3,
          rounded=True)
plt.title("Decision Tree Structure")
plt.show()

In [ ]:
y_train_pred = dt_model.predict(X_train_scaled)
y_val_pred = dt_model.predict(X_val_scaled)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_train, y_train_pred, color='blue', alpha=0.3)
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
plt.xlabel("Actual (Train)")
plt.ylabel("Predicted (Train)")
plt.title("Decision tree: Training Data")

plt.subplot(1, 2, 2)
plt.scatter(y_val, y_val_pred, color='green', alpha=0.3)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', lw=2)
plt.xlabel("Actual (Validation)")
plt.ylabel("Predicted (Validation)")
plt.title("Decision tree: Validation Data")
plt.show()

plt.tight_layout()
plt.show()

In [ ]:
# 3) XGBoost
xgb_model = XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=6, random_state=42)
results.append(evaluate_model(xgb_model, X_train_scaled, y_train, X_val_scaled, y_val, "XGBoost"))

In [ ]:
y_train_pred = xgb_model.predict(X_train_scaled)
y_val_pred = xgb_model.predict(X_val_scaled)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_train, y_train_pred, color='blue', alpha=0.3)
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
plt.xlabel("Actual (Train)")
plt.ylabel("Predicted (Train)")
plt.title("XGBoost: Training Data")

plt.subplot(1, 2, 2)
plt.scatter(y_val, y_val_pred, color='green', alpha=0.3)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', lw=2)
plt.xlabel("Actual (Validation)")
plt.ylabel("Predicted (Validation)")
plt.title("XGBoost: Validation Data")

plt.tight_layout()
plt.show()

In [ ]:
#4) LightGBM
lgbm_model = LGBMRegressor(n_estimators=200, learning_rate=0.1, max_depth=6, random_state=42, verbose=-1)
results.append(evaluate_model(lgbm_model, X_train_scaled, y_train, X_val_scaled, y_val, "LightGBM"))

In [ ]:
y_train_pred = lgbm_model.predict(X_train_scaled)
y_val_pred = lgbm_model.predict(X_val_scaled)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_train, y_train_pred, color='blue', alpha=0.3)
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
plt.xlabel("Actual (Train)")
plt.ylabel("Predicted (Train)")
plt.title("LightGBM: Training Data")

plt.subplot(1, 2, 2)
plt.scatter(y_val, y_val_pred, color='green', alpha=0.3)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', lw=2)
plt.xlabel("Actual (Validation)")
plt.ylabel("Predicted (Validation)")
plt.title("LightGBM: Validation Data")
plt.show()

plt.tight_layout()
plt.show()

In [ ]:
# 5) catboost
catboost_model = CatBoostRegressor(n_estimators=200, learning_rate=0.1, max_depth=6, random_state=42, verbose=0)
results.append(evaluate_model(catboost_model, X_train_scaled, y_train, X_val_scaled, y_val, "CatBoost"))


In [ ]:
y_train_pred =catboost_model.predict(X_train_scaled)
y_val_pred = catboost_model.predict(X_val_scaled)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_train, y_train_pred, color='blue', alpha=0.3)
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
plt.xlabel("Actual (Train)")
plt.ylabel("Predicted (Train)")
plt.title("catboost: Training Data")

plt.subplot(1, 2, 2)
plt.scatter(y_val, y_val_pred, color='green', alpha=0.3)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', lw=2)
plt.xlabel("Actual (Validation)")
plt.ylabel("Predicted (Validation)")
plt.title("catboost: Validation Data")
plt.show()

plt.tight_layout()
plt.show()

In [ ]:
results.append({"Model": "Linear Regression", "MAE": lr_mae, "MSE": lr_mse, "R2": lr_r2})
results.append({"Model": "Random Forest", "MAE": rf_mae, "MSE": rf_mse, "R2": rf_r2})

In [ ]:

all_comparison_df = pd.DataFrame(results)

all_comparison_df_sorted = all_comparison_df.sort_values(by='R2', ascending=False).reset_index(drop=True)

print("--- COMPREHENSIVE 7-MODELS COMPARISON ---")
display(all_comparison_df_sorted)

print("-" * 50)

best_model = all_comparison_df_sorted.loc[0, 'Model']
best_score = all_comparison_df_sorted.loc[0, 'R2']
print(f" The Best Model is: {best_model} (R2 = {best_score:.3f})")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = ['CatBoost', 'LightGBM', 'XGBoost', 'Decision Tree', 'KNN', 'Linear Regression', 'Random Forest']
mae = [5.95, 5.76, 5.91, 7.18, 8.40, 9.49, 11.15]
mse = [123.08, 127.28, 128.24, 166.91, 217.99, 289.74, 391.81]
r2 = [0.988, 0.988, 0.988, 0.984, 0.980, 0.973, 0.964]

x = np.arange(len(models))
width = 0.25

fig, ax1 = plt.subplots(figsize=(12, 6))
ax2 = ax1.twinx()

ax1.bar(x - width, mae, width, label='MAE', color='blue')
ax1.bar(x, mse, width, label='MSE', color='orange')
ax2.bar(x + width, r2, width, label='R2', color='green')

ax1.set_ylabel('MAE / MSE')
ax2.set_ylabel('R2 Score')
ax1.set_xticks(x)
ax1.set_xticklabels(models, rotation=15)
ax1.set_title('Comprehensive 7-Models Regression Comparison')

fig.tight_layout()
plt.show()

In [ ]:
import pandas as pd

new_product_data = {
    'Year': 2026,
    'Month': 9,
    'Season': 'Fall',
    'Is_Ramadan': 1,
    'Annual_Sin': 0.5,
    'Annual_Cos': 0.86,
    'Governorate': 'cairo',
    'Region': 'Northern',
    'Market_Type': 'Retail',
    'Unit': 'KG',
    'Urbanization': 0.96,
    'Commodity': 'Beans',
    'Category': 'Legumes',
    'Inflation_Index': 103.57,
    'Supply_Level': 76.10,
    'Demand_Level': 90.09,
    'Transport_Cost': 5.26,
    'Historical_Median': 50.49
}

new_df = pd.DataFrame([new_product_data])

new_encoded = pd.get_dummies(new_df, columns=['Season','Governorate','Region','Commodity','Market_Type','Category','Unit'])

new_encoded = new_encoded.reindex(columns=X_train.columns, fill_value=0)

new_scaled = scalar.transform(new_encoded)

predicted_price = catboost_model.predict(new_scaled)[0]

print("---  New Product Price Prediction ---")
print(f"Product: {new_product_data['Commodity']} in {new_product_data['Governorate']}")
print(f"Predicted Price: {predicted_price:.2f} EGP")